# 유지형 되뇌기(A) — 학습 (파일럿)

`11_되뇌기 구현 가이드.md` §3.2·§5, §6 기준. `notebooks/03_rehearsal_maintenance_prep.ipynb`가
만든 (input_ids, attention_mask, labels)로 `paust/pko-t5-small`을 지도학습한다.
파일럿(small)이고, 검증되면 `paust/pko-t5-base`로 스케일업한다.

평가지표:
- **ROUGE-L** — 참고용(추출요약 표준 지표)
- **novel n-gram 비율**(가이드 §6 조작 점검) — A는 이 값이 학습이 진행될수록
  0에 가까워져야 정상. 원문(=토큰화 전 input, 즉 rehearse_maintenance의
  `chunk.text`에 해당)과 비교하는 게 중요하다 — label과 비교하면 안 됨(label도
  이미 원문의 부분집합이라 "0에 가까움"이 당연해져서 검증 의미가 없어짐).
  그래서 `compute_metrics`가 아니라 별도 콜백(`NovelNGramCallback`)에서 검증셋
  일부를 직접 생성해 **원문과** 비교한다.

In [1]:
import os
import sys
from pathlib import Path

root = Path.cwd()
while not (root / "src").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root))

from src.notebook_setup import setup_project

setup_project()
API_KEY_SET = bool(os.environ.get("NVIDIA_NIM_API_KEY"))

import datasets
import evaluate
import numpy as np
import torch
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    TrainerCallback,
)

from src.pipeline.rehearsal import novel_ngram_ratio


project root: /Users/lucyroh/Desktop/STUDY/Data Projects/rehearse-then-recall
.env 로드: 성공 ✅
NVIDIA_NIM_API_KEY: 설정됨 ✅


/Users/lucyroh/Desktop/STUDY/Data Projects/rehearse-then-recall/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. 데이터/모델 로드

`03_rehearsal_maintenance_prep.ipynb`가 저장한 토큰화된 데이터셋(train
2,700 / val 300)을 불러온 뒤, 실제로 끝나는 규모로 서브샘플한다.

실측 확인: prep이 만든 데이터는 `input_ids` 길이가 중앙값 369토큰(90%가
876토큰 이하)인데도, 이 환경(MPS)에서 675스텝(2,700개 × batch_size 4)을
1시간 안에 못 끝냈다 — 스텝당 실제 속도가 예상보다 훨씬 느리다(평가 때
`predict_with_generate`의 자기회귀 생성 비용까지 포함). 그래서 이번 파일럿은
`PILOT_TRAIN_SIZE`/`PILOT_VAL_SIZE`로 확 줄인 규모로 코드 경로와 학습 자체가
되는지만 확인한다 — 스케일업은 나중에 이 값만 올리면 된다.

In [2]:
MODEL_NAME = "paust/pko-t5-small"
DATA_DIR = Path("data/processed/rehearsal_maintenance")
OUTPUT_DIR = "experiments/rehearsal_maintenance_small"
PILOT_TRAIN_SIZE = 200
PILOT_VAL_SIZE = 20

DATA_READY = (DATA_DIR / "train").exists() and (DATA_DIR / "val").exists()
if not DATA_READY:
    print(f"{DATA_DIR}에 토큰화된 데이터가 없습니다 — 03_rehearsal_maintenance_prep.ipynb를 먼저 실행하세요.")
else:
    full_train_dataset = datasets.Dataset.load_from_disk(str(DATA_DIR / "train"))
    full_val_dataset = datasets.Dataset.load_from_disk(str(DATA_DIR / "val"))

    train_dataset = full_train_dataset.select(range(min(PILOT_TRAIN_SIZE, len(full_train_dataset))))
    val_dataset = full_val_dataset.select(range(min(PILOT_VAL_SIZE, len(full_val_dataset))))
    print(f"전체 train {len(full_train_dataset)}행 중 {len(train_dataset)}행, "
          f"전체 val {len(full_val_dataset)}행 중 {len(val_dataset)}행 사용")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
    print(f"{MODEL_NAME} 로드 완료, 파라미터 수: {sum(p.numel() for p in model.parameters()):,}")


전체 train 2700행 중 200행, 전체 val 300행 중 20행 사용
paust/pko-t5-small 로드 완료, 파라미터 수: 95,628,672


## 2. 평가지표

`compute_metrics`는 ROUGE-L만 계산한다(예측 vs 정답 라벨, 표준 방식). novel
n-gram 비율은 아래 `NovelNGramCallback`에서 별도로 계산한다(원문과 비교해야
하므로 `compute_metrics`의 (예측, 라벨) 쌍만으로는 부족함 — 원문 `input_ids`가
필요).

In [3]:
rouge = evaluate.load("rouge")


def compute_metrics(eval_preds):
    predictions, labels = eval_preds
    if isinstance(predictions, tuple):
        predictions = predictions[0]
    predictions = np.where(predictions != -100, predictions, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=False)
    return {"rougeL": result["rougeL"]}


class NovelNGramCallback(TrainerCallback):
    """검증셋 일부로 novel n-gram 비율(가이드 §6)을 매 eval마다 로깅한다.
    A는 이 값이 학습이 진행될수록 0에 가까워져야 정상 — 원문(입력)과 비교하지,
    라벨과 비교하지 않는다(라벨도 이미 원문 부분집합이라 그거랑 비교하면
    항상 낮게 나와 검증 의미가 없어짐).
    """

    def __init__(self, tokenizer, eval_dataset, n: int = 3, sample_size: int = 5, max_new_tokens: int = 256):
        self.tokenizer = tokenizer
        self.examples = eval_dataset.select(range(min(sample_size, len(eval_dataset))))
        self.n = n
        self.max_new_tokens = max_new_tokens

    def on_evaluate(self, args, state, control, model=None, **kwargs):
        if model is None:
            return
        was_training = model.training
        model.eval()
        device = next(model.parameters()).device

        ratios = []
        for example in self.examples:
            input_ids = torch.tensor([example["input_ids"]]).to(device)
            with torch.no_grad():
                output_ids = model.generate(input_ids=input_ids, max_new_tokens=self.max_new_tokens)
            generated_text = self.tokenizer.decode(output_ids[0], skip_special_tokens=True)
            source_text = self.tokenizer.decode(example["input_ids"], skip_special_tokens=True)
            ratios.append(novel_ngram_ratio(generated_text, source_text, n=self.n))

        avg_ratio = sum(ratios) / len(ratios) if ratios else 0.0
        print(f"[novel {self.n}-gram 비율] step={state.global_step}: {avg_ratio:.4f} (검증 샘플 {len(ratios)}개)")

        if was_training:
            model.train()


## 3. 학습

`Seq2SeqTrainer` + `Seq2SeqTrainingArguments`. `predict_with_generate=True`가
필수인데(ROUGE-L 계산에 실제 생성 텍스트가 필요), 이 옵션이 없으면
`compute_metrics`가 raw logits을 받아서 ROUGE 계산이 안 된다.

체크포인트는 `experiments/rehearsal_maintenance_small/`에 저장된다(기존
`experiments/` 폴더 구조 재사용).

In [5]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    predict_with_generate=True,
    generation_max_length=256,
    load_best_model_at_end=True,
    metric_for_best_model="rougeL",
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[NovelNGramCallback(tokenizer, val_dataset)],
)

trainer.train()

                                              
  6%|▌         | 3/50 [15:07<10:58, 14.00s/it]    

{'loss': 1.3103, 'grad_norm': 2.7696285247802734, 'learning_rate': 4e-05, 'epoch': 0.2}


                                              
  6%|▌         | 3/50 [18:32<10:58, 14.00s/it] 

{'loss': 1.0522, 'grad_norm': 6072.04541015625, 'learning_rate': 3e-05, 'epoch': 0.4}


                                              
  6%|▌         | 3/50 [20:53<10:58, 14.00s/it] 

...

{'eval_loss': 0.45296794176101685, 'eval_rougeL': 0.08535256410256412, 'eval_runtime': 67.6789, 'eval_samples_per_second': 0.296, 'eval_steps_per_second': 0.074, 'epoch': 1.0}


[novel 3-gram 비율] step=50: 0.3935 (검증 샘플 5개)


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].
                                              
100%|██████████| 50/50 [24:37<00:00, 29.55s/it]

{'train_runtime': 1477.5656, 'train_samples_per_second': 0.135, 'train_steps_per_second': 0.034, 'train_loss': 0.9792593765258789, 'epoch': 1.0}


TrainOutput(global_step=50, training_loss=0.9792593765258789, metrics={'train_runtime': 1477.5656, 'train_samples_per_second': 0.135, 'train_steps_per_second': 0.034, 'total_flos': 64206545485824.0, 'train_loss': 0.9792593765258789, 'epoch': 1.0})

## 4. 최종 모델 저장

In [6]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"저장 완료: {OUTPUT_DIR}")

저장 완료: experiments/rehearsal_maintenance_small


## 5. 검증 — `sample01.txt` 청크로 실제 되뇌기 실행

지금까지 학습한 모델을 `rehearse_maintenance`(query-agnostic, `src/pipeline/rehearsal.py`)에
직접 넣어서, `paginate_semantic`이 만든 실제 청크에 대해 압축이 되는지, novel
n-gram 비율이 낮은지 확인한다.

In [ ]:
if not DATA_READY:
    print("데이터가 없어 검증을 건너뜁니다.")
elif not API_KEY_SET:
    print("NVIDIA_NIM_API_KEY가 없어(verbatim 스냅에 임베딩 API 필요) 검증을 건너뜁니다.")
else:
    import yaml

    from src.pipeline.chuncking import paginate_semantic, plain_text_to_paragraphs
    from src.pipeline.embeddings import embed_texts, load_config
    from src.pipeline.rehearsal import rehearse_maintenance

    with open("data/sample/sample01.txt", encoding="utf-8") as f:
        raw_text = f.read()

    paragraphs = plain_text_to_paragraphs(raw_text)
    chunk_cfg = yaml.safe_load(open("configs/chunking.yaml", encoding="utf-8"))
    embed_cfg = load_config("configs/importance_filter.yaml")

    chunks = paginate_semantic(
        paragraphs,
        min_words=chunk_cfg["min_words"],
        max_words=chunk_cfg["max_words"],
        granularity="paragraph",
        config=embed_cfg,
        embed_fn=embed_texts,
    )
    sample_chunks = chunks[:5]
    print(f"검증용 청크 {len(sample_chunks)}개 (전체 {len(chunks)}개 중 앞부분)")

    rehearsed = rehearse_maintenance(
        sample_chunks, model=model, tokenizer=tokenizer, embed_cfg=embed_cfg, embed_fn=embed_texts
    )

    ngram_ratios = []
    for original, compressed in zip(sample_chunks, rehearsed):
        ratio = novel_ngram_ratio(compressed.text, original.text, n=3)
        ngram_ratios.append(ratio)
        compression = len(compressed.text) / max(1, len(original.text))
        print(f"\n[청크 {original.index}] 원문 {len(original.text)}자 -> 압축본 {len(compressed.text)}자 (압축률 {compression:.1%})")
        print(f"  novel 3-gram 비율: {ratio:.4f}")
        print(f"  압축본: {compressed.text[:200]}")

    print(f"\n평균 novel 3-gram 비율: {sum(ngram_ratios) / len(ngram_ratios):.4f} (0에 가까울수록 정상)")

검증용 청크 5개 (전체 37개 중 앞부분)

[청크 0] 원문 692자 -> 압축본 836자 (압축률 120.8%)
  novel 3-gram 비율: 0.2093
  압축본: 둔탁한 정육면체 프레임을 가진 1호. 높게 치솟은 2호. 정글짐처럼 복잡하게 얽혀 있는 3호와 고고하게 양팔을 흔드는 4호. 모두 버려진 의자와 폐자재를 조립해 만든 구조물이었다. 노인 역시 윤재만큼이나 최선을 다해 의자를 선별하고 있다 노인 역시 윤재만큼이나 최선을 다해 의자를 선별하고 있다 노인 역시 윤재만큼이나 최선을 다해 의자를 선별하고 있다 노인 

[청크 1] 원문 928자 -> 압축본 1547자 (압축률 166.7%)
  novel 3-gram 비율: 0.1905
  압축본: 혼자만 먹기는 눈치가 보였는지 물어도 보지 않고 윤재와 장 주무관 몫까지 꼭 세 개를 시켰다. “계산은 나중에 오시는 분이 해주실 거예요.” 한두 번이 아니었으니 직원도 군말 없이 돌아서서 커피를 내리기 시작했다. 저희야 누구든 와주기만 해도 고마운 일이긴 한데…… 혼자만 먹기는 눈치가 보였는지 물어도 보지 않고 윤재와 장 주무관 몫까지 꼭 세 개를 시켰다

[청크 2] 원문 1083자 -> 압축본 111자 (압축률 10.2%)
  novel 3-gram 비율: 0.1538
  압축본: 그럼에도 윤재는 청령시가 그의 의자 프로젝트를 진행하기에 적합한 곳인지 잠시 고민할 수밖에 없었다. 그럼에도 윤재는 청령시가 그의 의자 프로젝트를 진행하기에 적합한 곳인지 잠시 고민할 수밖에 없었다.

[청크 3] 원문 783자 -> 압축본 60자 (압축률 7.7%)
  novel 3-gram 비율: 0.0000
  압축본: 윤재의 귀가 화끈거렸다. “바쁘시면 오늘은 먼저 가보셔도 돼요. 저는 작가님이랑 따로 할 이야기가 있어서.”

[청크 4] 원문 733자 -> 압축본 435자 (압축률 59.3%)
  novel 3-gram 비율: 0.4444
  압축본: 의자는 항상 새벽을 틈타 사라졌다. 의자는 항상 새벽을 틈타

## 정리

- 이번 파일럿은 법률문서 프로젝트(AIHub) 중 `PILOT_TRAIN_SIZE`/`PILOT_VAL_SIZE`
  (기본 200/20)개, `paust/pko-t5-small`, 1 epoch 기준이다 — 이 환경(MPS)에서
  전체 2,700개·2 epoch가 1시간 안에도 안 끝나는 걸 실측으로 확인해서 줄인
  것이다. 코드 경로 검증 목적이지 최종 품질 검증이 아니다.
- 다음 단계: (1) `PILOT_TRAIN_SIZE`를 올려 스케일업(단, 속도 먼저 실측하고
  올릴 것), (2) `paust/pko-t5-base`로 백본 교체, (3) novel n-gram 비율이
  기대만큼 안 떨어지면 verbatim 후처리(`_snap_to_verbatim`) 자체가 제대로
  스냅하고 있는지 개별 샘플로 디버깅, (4) 검증 결과를 Obsidian
  `11_되뇌기 구현 가이드.md`에 반영.